In [ ]:
!pip install -q gradio kagglehub pandas scikit-learn

In [ ]:
import os
import pandas as pd
import numpy as np
import gradio as gr
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
print("🎬 Fetching TMDB Movie Dataset from Kaggle...")
path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")
csv_path = os.path.join(path, "tmdb_5000_movies.csv")

🎬 Fetching TMDB Movie Dataset from Kaggle...
Using Colab cache for faster access to the 'tmdb-movie-metadata' dataset.


In [ ]:
df_movies = pd.read_csv(csv_path)

In [ ]:
movie_features = ['budget', 'popularity', 'runtime']
X_movie = df_movies[movie_features]
y_movie = df_movies['revenue']

In [ ]:
valid_movie_rows = X_movie.notna().all(axis=1) & y_movie.notna()
X_movie_clean = X_movie[valid_movie_rows]
y_movie_clean = y_movie[valid_movie_rows]

In [ ]:
movie_model = LinearRegression()
movie_model.fit(X_movie_clean, y_movie_clean)
print("🎉 Movie Box Office Predictor Trained Successfully!")

🎉 Movie Box Office Predictor Trained Successfully!


In [ ]:
def predict_box_office(movie_name, budget_in_millions, hype_level, runtime):
    # Multiply the user's input by 1 Million so the model understands the raw number
    actual_budget = budget_in_millions * 1000000

    # Format the inputs for our model (The model ignores the name string, uses the numbers)
    features = np.array([[actual_budget, hype_level, runtime]])
    predicted_revenue = movie_model.predict(features)[0]

    # Floor at 0 so we don't display negative values
    final_revenue = max(0, predicted_revenue)
    revenue_in_millions = final_revenue / 1000000

    # Dynamic text output utilizing the Movie Name!
    return (
        f"🎬 Film: {movie_name}\n"
        f"📊 Predicted Global Revenue: ${revenue_in_millions:.2f} Million\n"
        f"💰 Exact Calculated Total: ${int(final_revenue):,}"
    )

In [ ]:
movie_interface = gr.Interface(
    fn=predict_box_office,
    inputs=[
        gr.Textbox(label="🎬 Movie Title / Project Name", placeholder="e.g., Avatar 3"),
        gr.Number(label="💰 Production Budget (In Millions of $)", value=50, info="Just type the number (e.g., 50 means $50,000,000)"),
        gr.Slider(minimum=1, maximum=300, value=50, label="🔥 Internet Hype Level", info="1 = Low Indie Buzz, 300 = Global Blockbuster Hype"),
        gr.Number(label="⏳ Runtime (Minutes)", value=120)
    ],
    outputs=gr.Textbox(label="🎉 Box Office Forecast Result", lines=4),
    title="🍿 The Festive Box Office Revenue Predictor (Task 1)",
    description="Welcome to the Film Festival Simulator! Give your movie a name, set your budget in millions, gauge the internet buzz, and project your global box office earnings instantly."
)

In [ ]:
movie_interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0bc24465caa9880870.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
